# Zep 记忆（Zep Memory）

> **一个生产级记忆服务：从智能体的对话中提取事实、分类对话、识别实体，并构建时序知识图谱。**

想象雇一位私人助理，旁听你的每一场会议。这位助理不会逐字记录对话，而是把关键事实写在索引卡片上（「客户的预算是 5 万美元」），记下提到了谁（「市场部的 Alice」），给每场会议标注主题（「第三季度规划」），再画一张图，展示人和项目之间如何关联。这一切都在后台完成，你只管继续交谈。

**Zep** 为你的 AI 智能体做的就是这件事。你把消息发给 Zep，后台流程（pipeline）处理剩下的工作。**对话分类器**给每个轮次标注意图和主题。**实体抽取器**识别人物、组织和项目。**事实提取器**从非结构化文本中提炼结构化陈述。**知识图谱构建器**（由 Graphiti 驱动）绘制实体之间随时间变化的关系。

你的智能体不必等待这些处理完成。Zep 在后台工作，智能体继续响应用户。最终你得到一个记忆层，支持三种检索模式：对已提取事实的**语义检索**、对实体关系的**图遍历**，以及可直接放进提示词的**组装上下文**（assembled context）字符串。

**学完本 notebook，你将理解：**
- Zep 的后台流程如何把原始对话变成结构化记忆。
- 如何用 Zep SDK 添加消息、搜索事实、检索上下文。
- 如何构建一个用 Zep 实现持久化跨会话记忆的智能体循环。
- 什么时候该选 Zep，什么时候更简单的方案就够用。

## 关键概念

- **线程（thread）**：Zep 中的对话容器。每个线程有唯一 ID，属于某个用户。可以把它看成一场聊天会话。线程记录消息历史，以及从这段历史中抽取的全部知识。
- **后台流程（background pipeline）**：你添加消息后自动运行的处理过程。它们提取事实、识别实体、分类对话、构建图关系。你不会直接调用它们，它们异步运行（在后台执行，不阻塞你的代码）。
- **事实提取（fact extraction）**：把非结构化对话转成结构化陈述。例如「我去年三月搬到了伦敦」会变成一条带时间戳、可查询的记录。事实是 Zep 语义记忆的基本单元。
- **实体抽取（entity extraction）**：跨对话识别命名实体（人物、地点、组织、产品）。Zep 对实体去重（合并重复项）并跨线程关联，为智能体学到的一切建立一份持久的登记表。
- **对话分类（dialog classification）**：按意图（提问、指令、反馈）、主题和情感自动标注每个对话轮次。无需人工标注，也无需训练自定义模型。
- **时序知识图谱（temporal knowledge graph）**：一个图数据库（由 Graphiti 驱动），带时间戳地存储实体之间的关系。「时序」指它记录事物*在什么时候*为真。你可以问「上周是什么情况」，观察知识如何变化。
- **图搜索（graph search）**：用自然语言查询 Zep 的知识图谱。结果按语义相关性排序（即你的查询与已存事实在含义上的接近程度）。你可以在边（事实）、节点（实体）或片段（episode，对话片段）上搜索。
- **组装上下文（assembled context）**：Zep 用相关事实、实体和消息预先生成的一段文本。你直接把它注入 LLM 的系统提示词即可，无需自己写检索逻辑。

## 架构

<p align="center">
  <img src="../../images/diagrams/27_zep_memory.svg" alt="Zep 记忆架构图" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart TD
    subgraph App["Agent Application"]
        A["User Messages"] --> B["Zep SDK\n(add_messages)"]
        K["Zep SDK\n(graph.search)"] --> L["Agent Context\nWindow"]
    end

    subgraph ZepService["Zep Service"]
        B --> C["Message Store"]
        C --> D["Background Pipelines"]

        subgraph Pipelines["Async Processing"]
            D --> E["Dialog\nClassifier"]
            D --> F["Entity\nExtractor"]
            D --> G["Fact\nExtractor"]
            D --> H["Knowledge Graph\nBuilder (Graphiti)"]
        end

        subgraph Storage["Zep Storage"]
            I[("Messages +\nFacts +\nEntities +\nGraph")]
        end

        E --> I
        F --> I
        G --> I
        H --> I

        subgraph Search["Search API"]
            J["Semantic Search\n+ Graph Traversal\n+ Temporal Filter"]
        end

        I --> J
    end

    J --> K

    style I fill:#4f46e5,color:#fff
    style E fill:#059669,color:#fff
    style F fill:#059669,color:#fff
    style G fill:#059669,color:#fff
    style H fill:#8b5cf6,color:#fff
    style J fill:#d97706,color:#fff
    style L fill:#6366f1,color:#fff
```

</details>

**数据在 Zep 中的流动：**

1. 你的智能体通过 SDK 的 `add_messages()` 方法，把用户和助手消息发给 Zep。
2. Zep 立即存储消息，后台流程开始处理。
3. 对话分类器给每个轮次打标签。实体抽取器识别实体并去重。事实提取器提炼结构化事实。知识图谱构建器更新 Graphiti 时序图。
4. 所有抽取出的数据都落入 Zep 的统一存储层。
5. 生成回复之前，你的智能体调用 `graph.search()` 或 `get_user_context()`，Zep 返回排好序的事实和相关实体。
6. 你的智能体把检索到的记忆注入上下文窗口，给出个性化的回复。

## 环境准备

安装 Zep Cloud SDK 和 Anthropic SDK。你还需要 `python-dotenv` 来从 `.env` 文件加载 API 密钥。

In [ ]:
%pip install -q zep-cloud python-dotenv deepagents langchain-openai

加载环境变量，初始化 Zep 和 Anthropic 客户端。你需要两个 API 密钥：

- `ZEP_API_KEY`：你的 Zep Cloud API 密钥（在 [getzep.com](https://www.getzep.com?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) 注册）
- `LLM_API_KEY`：你的 Anthropic API 密钥

In [ ]:
import os
import time
import uuid

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)



def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


## 实现

我们分五步构建一个带记忆增强的智能体：

1. 在 Zep 中创建用户和线程。
2. 添加对话消息，交给后台流程处理。
3. 用图搜索查找已提取的事实。
4. 检索组装上下文，用于注入提示词。
5. 把所有环节接进一个可复用的智能体循环。

### 第 1 步：创建用户和线程

Zep 中的每段对话都属于某个用户，存放在一个线程里。我们生成唯一 ID，这样本 notebook 可以安全地重复运行。

In [ ]:
user_id = f"demo_user_{uuid.uuid4().hex[:8]}"
thread_id = f"demo_thread_{uuid.uuid4().hex[:8]}"

zep.user.add(
    user_id=user_id,
    first_name="Alice",
    metadata={"role": "ml_engineer"},
)
zep.thread.create(thread_id=thread_id, user_id=user_id)

print(f"Created user:   {user_id}")
print(f"Created thread: {thread_id}")


### 第 2 步：添加消息

我们向 Zep 发送一段真实的多轮对话。调用 `add_messages()` 之后，Zep 存储消息并启动后台流程。这些流程提取事实、识别实体、分类对话意图，并更新知识图谱。

处理需要几秒钟。我们加入一段短暂等待，确保查询时结果已经就绪。

In [ ]:
conversation = [
    Message(role="user", content="Hi! I'm Alice, a machine learning engineer at Acme Corp."),
    Message(role="assistant", content="Nice to meet you, Alice! What are you working on at Acme Corp?"),
    Message(role="user", content="We're building a recommendation system. My team lead is Bob Chen."),
    Message(role="assistant", content="Interesting! What kind of recommendations?"),
    Message(role="user", content="Product recommendations for our e-commerce platform. We use Python and PyTorch."),
    Message(role="assistant", content="Great choices. Are you using collaborative filtering or content-based approaches?"),
    Message(role="user", content=(
        "We started with collaborative filtering but we're moving to a hybrid approach. "
        "Our budget for this quarter is $200k."
    )),
    Message(role="assistant", content="A hybrid approach often gives better results. How large is your dataset?"),
    Message(role="user", content="About 50 million user-product interactions. We also have rich product metadata."),
]

zep.thread.add_messages(thread_id=thread_id, messages=conversation)
print(f"Added {len(conversation)} messages to thread '{thread_id}'")

# 等待后台流程处理完成
print("Waiting for Zep to extract facts and entities...")
time.sleep(10)
print("Processing complete.")


### 第 3 步：搜索已提取的事实

Zep 的图搜索让你用自然语言查询知识图谱。用 `scope="edges"` 搜索时，得到的是提取出的事实（实体之间的关系）。每条事实都包含原始陈述和一个相关性分数。

In [ ]:
queries = [
    "What technology does Alice use?",
    "Who is on Alice's team?",
    "What is the project budget?",
]

for query in queries:
    results = zep.graph.search(
        query=query,
        user_id=user_id,
        scope="edges",
        limit=3,
    )

    print(f"Query: {query}")
    for edge in results.edges or []:
        print(f"  -> {edge.fact}")
    print()


### 第 4 步：检索组装上下文

你也可以不手动搜索，直接向 Zep 要一份预组装的上下文字符串。它把相关事实、实体和最近消息合并成一段文本，你可以直接注入 LLM 的系统提示词。

这是生产环境最常见的模式，省去自己编写检索和排序逻辑。

In [ ]:
context_response = zep.thread.get_user_context(thread_id=thread_id)

print("Assembled context for LLM injection:")
print("=" * 60)
print(context_response.context or "(No context available yet)")


### 第 5 步：构建记忆增强的智能体循环

现在把所有环节组合成一个可复用的智能体类。模式如下：

1. 用户发送一条消息。
2. 我们把它添加到 Zep，并请求返回相关上下文（用 `return_context=True`）。
3. 我们构建包含上下文的系统提示词。
4. 我们调用 LLM。
5. 我们把助手的回复存入 Zep，供将来参考。

因为 Zep 在**用户层级**存储记忆，同一个用户的事实可以跨线程传递。你的智能体能跨对话记住一切。

In [ ]:
class ZepMemoryAgent:
    """使用 Zep 实现持久化跨会话记忆的智能体。"""

    def __init__(
        self,
        zep_client,
        model,
        user_id,
        thread_id,
    ):
        self.zep = zep_client
        self.model = model
        self.user_id = user_id
        self.thread_id = thread_id

    def chat(self, user_input):
        """发送一条消息，返回带记忆增强的回复。"""
        # 在一次调用中添加用户消息并取回相关上下文
        add_response = self.zep.thread.add_messages(
            thread_id=self.thread_id,
            messages=[Message(role="user", content=user_input)],
            return_context=True,
        )
        context = add_response.context or ""

        # 用 Zep 组装好的上下文构建系统提示词
        system_prompt = (
            "You are a helpful assistant with memory of past conversations.\n\n"
            "Use the following context to personalize your response. "
            "Reference relevant facts naturally without mentioning the memory system.\n\n"
            f"<context>\n{context}\n</context>"
        )

        # Zep 组装的上下文每轮变化，所以每轮重建智能体
        agent = create_deep_agent(model=self.model, system_prompt=system_prompt)
        result = agent.invoke({"messages": [{"role": "user", "content": user_input}]})
        assistant_text = to_text(result["messages"][-1].content)

        # 把助手的回复存入 Zep
        self.zep.thread.add_messages(
            thread_id=self.thread_id,
            messages=[Message(role="assistant", content=assistant_text)],
        )

        return assistant_text


## 运行示例

我们为同一个用户创建一个**新线程**。虽然这是一场全新的对话，智能体仍能访问上一个线程中的事实。这就是 Zep 跨会话记忆的实际效果。

In [ ]:
agent_thread_id = f"agent_thread_{uuid.uuid4().hex[:8]}"
zep.thread.create(thread_id=agent_thread_id, user_id=user_id)

agent = ZepMemoryAgent(
    zep_client=zep,
    llm_client=llm,
    user_id=user_id,
    thread_id=agent_thread_id,
)

# 智能体能回忆起第一个线程里的事实
exchanges = [
    "What do you remember about my work?",
    "Who's my team lead?",
    "What's our quarterly budget?",
    "Based on what you know about our project, what papers should I read?",
]

for msg in exchanges:
    print(f"User:  {msg}")
    reply = agent.chat(msg)
    print(f"Agent: {reply}\n")


### 对比检索模式

Zep 支持不同的搜索范围（scope），每种范围返回已存知识的不同视图：

- **edges**：提取出的事实（实体之间的关系）。最适合直接回忆。
- **nodes**：实体摘要。最适合「介绍一下 X」这类问题。
- **auto**：Zep 为你的查询自动选择最佳范围。

我们用同一个查询跑一遍每种范围，看看区别。

In [ ]:
query = "What is Alice working on?"
print(f"Query: '{query}'\n")

# 事实搜索（edges）
fact_results = zep.graph.search(
    query=query, user_id=user_id, scope="edges", limit=3,
)
print("Facts (edges):")
for edge in fact_results.edges or []:
    print(f"  - {edge.fact}")

# 实体搜索（nodes）
node_results = zep.graph.search(
    query=query, user_id=user_id, scope="nodes", limit=3,
)
print("\nEntities (nodes):")
for node in node_results.nodes or []:
    print(f"  - {node.name}: {node.summary}")

# 自动搜索（Zep 自行选择最佳 scope）
auto_results = zep.graph.search(
    query=query, user_id=user_id, scope="auto", limit=5,
)
print("\nAuto (Zep chooses):")
if auto_results.edges:
    for edge in auto_results.edges:
        print(f"  Fact: {edge.fact}")
if auto_results.nodes:
    for node in auto_results.nodes:
        print(f"  Entity: {node.name}")


清理本次演示创建的资源。

In [ ]:
zep.thread.delete(thread_id=thread_id)
zep.thread.delete(thread_id=agent_thread_id)
zep.user.delete(user_id=user_id)
print(f"Deleted user '{user_id}' and associated threads.")


## 权衡

### Zep 的优势场景

- **多会话智能体**：Zep 在用户层级存储记忆，一段对话的事实可以带入下一段。你不必自己管理记忆持久化。
- **自动抽取**：你只管发送原始消息。Zep 提取事实、识别实体、分类对话、构建知识图谱。无需自定义 NLP 流程。
- **托管基础设施**：云版本负责扩缩容、索引和模型更新。你的团队专注智能体逻辑，不用操心记忆基础设施。
- **时间感知**：知识图谱记录事实在何时为真。你的智能体能区分「用户曾住在纽约」（过去）和「用户住在伦敦」（现在）。

### 什么时候会出问题

- **抽取准确性**：后台流程用 LLM 提取事实，可能漏掉细微差别、虚构事实，或把实体归错类。抽取模型不容易调优。
- **首次查询的延迟**：抽取是异步的。消息添加后，事实不会立刻可查。对实时应用来说，新知识可被搜索前有一小段延迟。
- **规模化成本**：每条消息都会触发基于 LLM 的抽取。消息量大时，单条消息的抽取成本会累积。更简单的技术（如缓冲或窗口记忆）每条消息更便宜。
- **供应商依赖**：云版本需要 Zep 账号和 API 密钥。开源版本需要用特定基础设施自行部署。无论选哪种，你都要接受 Zep 的数据模型。

## 延伸阅读

- [Zep 文档](https://docs.getzep.com?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - 官方文档，覆盖线程、记忆搜索、事实提取和 SDK 集成。
- [Zep GitHub 仓库](https://github.com/getzep/zep) - 开源代码，支持自托管部署和社区贡献。
- [Zep Python SDK（PyPI）](https://pypi.org/project/zep-cloud/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - 包详情、版本历史和安装说明。
- [Graphiti：时序知识图谱](https://github.com/getzep/graphiti) - 驱动 Zep 知识图谱层的图引擎。
- [Anthropic：构建高效智能体](https://www.anthropic.com/engineering/building-effective-agents?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - 与 Zep 记忆能力互补的智能体架构模式。

*← 上一篇：[26 - Letta (MemGPT) 模式](../26_letta_memgpt_patterns/) · 下一篇：[28 - 记忆评估](../28_memory_evaluation/) →*

## 🧪 动手练习

三个小挑战，帮你加深理解。每个大约需要 10-30 分钟。

### 挑战 1：事实提取审计
向一个线程添加 10 条消息，等待后台处理完成，然后用 `graph.search()` 取回提取出的事实。把这些事实与手工整理的消息基准真值列表对比，计算抽取的查准率和查全率。

### 挑战 2：检索方法对比
对同一组查询，对比 `graph.search()` 和 `get_user_context()` 的结果。衡量哪种方法返回的信息更相关，并记录两者的延迟差异。把结果整理成一张对比表。

### 挑战 3：多线程主题路由
在 Zep 中创建 3 个独立线程，各自对应一个主题（如工作、爱好、旅行）。向每个线程添加相关消息，再跨三个线程查询，衡量主题隔离是否提升了检索查准率。这与 17 记忆路由中的路由模式相关。

![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--27-zep-memory--zep-memory)
